# Proyecto Final — Problema 1: Word Cloud

**Curso:** Text Mining & Image Recognition  
**Objetivo:** encontrar los 3 usuarios más mencionados, construir un corpus para cada uno y analizar por qué son citados.

> El dataset original no se incluye en el repositorio porque es grande. Coloque `tw_source.csv.zip` en la raíz del proyecto o súbalo a Google Colab cuando ejecute el Notebook.


## Datos

El archivo corresponde al conocido formato de seis columnas del dataset suministrado:

1. `sentiment`
2. `id`
3. `timestamp`
4. `query`
5. `author`
6. `text`

El archivo no trae encabezado, por lo que se asignan estos nombres al leerlo.


In [ ]:
# En Google Colab, si hace falta:
# !pip install pandas nltk wordcloud matplotlib

import re, zipfile
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud

import nltk
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

COLUMNAS = ["sentiment","id","timestamp","query","author","text"]
ARCHIVO = Path("tw_source.csv.zip")

if not ARCHIVO.exists():
    try:
        from google.colab import files
        print("Suba tw_source.csv.zip")
        uploaded = files.upload()
        ARCHIVO = Path(next(iter(uploaded)))
    except Exception:
        raise FileNotFoundError("Coloque tw_source.csv.zip junto al Notebook.")


## Paso 1 — Encontrar los usuarios más populares

Interpretamos **popular** como el usuario que aparece más veces mencionado mediante `@usuario`, porque el ejercicio luego pregunta por qué **citan** a esos usuarios.

Para no cargar 1.6 millones de tweets de una sola vez, se procesa el CSV en bloques de 100,000 filas.


In [ ]:
patron_mencion = re.compile(r"@([A-Za-z0-9_]+)")
conteo = Counter()

with zipfile.ZipFile(ARCHIVO) as z, z.open("tw_source.csv") as f:
    for bloque in pd.read_csv(
        f, header=None, names=COLUMNAS, encoding="latin-1",
        chunksize=100_000
    ):
        for texto in bloque["text"].fillna("").astype(str):
            conteo.update(x.lower() for x in patron_mencion.findall(texto))

top3 = conteo.most_common(3)
top3


### Resultado

Los tres usuarios más mencionados son:

| Posición | Usuario | Menciones |
|---|---|---:|
| 1 | `@mileycyrus` | 4,580 |
| 2 | `@tommcfly` | 3,904 |
| 3 | `@ddlovato` | 3,474 |


## Paso 2 — Construcción de los tres corpus

Cada corpus contiene:
- **Content:** Tweet completo.
- **ID:** identificador del tweet.
- **Timestamp:** fecha/hora.
- **Length:** cantidad de caracteres del tweet.


In [ ]:
usuarios = [u for u, _ in top3]
corpus = {u: [] for u in usuarios}

with zipfile.ZipFile(ARCHIVO) as z, z.open("tw_source.csv") as f:
    for bloque in pd.read_csv(
        f, header=None, names=COLUMNAS, encoding="latin-1",
        chunksize=100_000
    ):
        texto_lower = bloque["text"].fillna("").astype(str).str.lower()

        for usuario in usuarios:
            mascara = texto_lower.str.contains(
                rf"(?<!\w)@{re.escape(usuario)}\b", regex=True
            )
            if mascara.any():
                tmp = bloque.loc[mascara, ["id","timestamp","text"]].copy()
                tmp["length"] = tmp["text"].astype(str).str.len()
                tmp = tmp.rename(columns={"text":"content"})
                corpus[usuario].append(tmp)

for usuario in usuarios:
    corpus[usuario] = (
        pd.concat(corpus[usuario], ignore_index=True)
          .drop_duplicates(subset=["id"])
    )
    corpus[usuario].to_csv(f"corpus_{usuario}.csv", index=False)
    print(usuario, len(corpus[usuario]))


## Paso 3 — Extraer contexto y limpiar texto

Para saber **por qué citan al usuario**, tomamos hasta 8 palabras antes y 8 después de cada mención.

Luego:
1. Quitamos URLs y otras menciones.
2. Quitamos stopwords.
3. Aplicamos **stemming**.
4. Aplicamos **lemmatización**.
5. Contamos las palabras resultantes.


In [ ]:
stop = set(stopwords.words("english"))
stop.update({
    "im","ive","dont","cant","youre","thats","got","get","going",
    "amp","rt","lol","quot","http","https","www"
})

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

token_re = re.compile(r"@[\w_]+|https?://\S+|www\.\S+|[A-Za-z]+(?:'[A-Za-z]+)?")

def contexto_usuario(texto, usuario, ventana=8):
    tokens = token_re.findall(str(texto))
    salida = []
    for i, token in enumerate(tokens):
        if token.lower() == f"@{usuario}":
            ventana_tokens = tokens[max(0,i-ventana):i] + tokens[i+1:i+ventana+1]
            for palabra in ventana_tokens:
                p = palabra.lower().strip("'")
                if p.startswith("@") or p.startswith("http") or p in stop or len(p) < 3:
                    continue
                # Stemming y luego lematización, como solicita el ejercicio
                stem = stemmer.stem(p)
                lemma = lemmatizer.lemmatize(p)
                salida.append((p, stem, lemma))
    return salida

frecuencias = {}

for usuario in usuarios:
    original = Counter()
    stems = Counter()
    lemmas = Counter()

    for tweet in corpus[usuario]["content"]:
        for palabra, stem, lemma in contexto_usuario(tweet, usuario):
            original[palabra] += 1
            stems[stem] += 1
            lemmas[lemma] += 1

    frecuencias[usuario] = {
        "original": original,
        "stem": stems,
        "lemma": lemmas
    }

    print("\n@", usuario)
    print("Top lemas:", lemmas.most_common(10))


## Paso 4 — WordCloud Top 10

El WordCloud se limita a las 10 palabras lematizadas más frecuentes del contexto de cada usuario.


In [ ]:
for usuario in usuarios:
    top10 = dict(frecuencias[usuario]["lemma"].most_common(10))

    wc = WordCloud(
        width=1400, height=800,
        background_color="white",
        collocations=False
    ).generate_from_frequencies(top10)

    plt.figure(figsize=(12,6))
    plt.imshow(wc, interpolation="bilinear")
    plt.title(f"Top 10 contexto de @{usuario}")
    plt.axis("off")
    plt.show()

    wc.to_file(f"wordcloud_{usuario}.png")


## Interpretación — ¿Por qué citan a cada usuario?

### @mileycyrus
Las palabras cercanas muestran principalmente afecto y apoyo ('love', 'good', 'hope'), junto con promoción/votación ('vote') y referencias a su trabajo ('movie'). Esto sugiere que muchas menciones son mensajes de fans, apoyo y promoción.

### @tommcfly
Predominan referencias directas a Tom, expresiones de afecto ('love'), conversación ('say', 'think'), llamados al grupo ('guys') y referencias a Brasil. El contexto sugiere interacción de fans, conversación y seguimiento internacional.

### @ddlovato
Aparecen con alta frecuencia 'demi', 'love', 'wish', 'hope', 'wait' y 'awesome'. El contexto apunta principalmente a mensajes de fans que expresan cariño, deseos, expectativa y apoyo.

> Estas interpretaciones describen el contexto lingüístico observado en este dataset; no pretenden explicar la intención individual de cada tweet.


## Conclusión

El análisis de menciones permite identificar los usuarios con mayor presencia en el dataset. El análisis del contexto, combinado con limpieza, stemming, lematización y frecuencia de términos, permite resumir los temas y expresiones que rodean a cada usuario.
